# Study report: code-generation

The study measures how four code generation variants change the IRQ
entry, the ISR to thread wake and the thread to thread latency:
`speed-opt` (`SPEED_OPTIMIZATIONS=y`), `assert` (`ASSERT=y`),
`assert-spin` (`ASSERT=y`, `SPIN_VALIDATE=y`) and `fpu-sharing`
(`FPU=y`, `FPU_SHARING=y`). The study imports the three experiments of
the parent study kconfig-factors.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-kconfig-factors@nucleo_g474re/st-code-generation"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts"))
from probatrix.benchmarks import data  # noqa: E402
from probatrix.benchmarks import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## Background

- `speed-opt`: the compiler optimizes for speed, not for size. The
  hardware does most of the IRQ entry, so the compiler has little code
  to change there.
- `assert`: `ASSERT` adds checks to the kernel code, also in the
  sections with locked interrupts.
- `assert-spin`: `SPIN_VALIDATE` adds a check to each spinlock pair.
  The checks of `assert-spin` on the wake paths compare it with
  `assert`.
- `fpu-sharing`: the kernel keeps an FP context for each thread. PendSV
  examines the FP bit of `EXC_RETURN`. The IRQ entry path saves no FP
  context.

The checks of the form `ratio_to` give a range of the ratio to the
baseline. The figures give this range as a difference, with the value
of the baseline of the same run.

In [ ]:
LOAD_NAMES = {"none": "no load", "busy16": "16 busy threads", "t64": "64 load timers"}


def diffs(exp):
    """Each point minus the point of the variant baseline with the same load, in the same run (nbfig.diffs)."""
    return nbfig.diffs(inst, exp, "variant", "baseline", match=("load",), names=LOAD_NAMES)

VARIANTS = ['speed-opt', 'assert', 'assert-spin', 'fpu-sharing']

## How to read the difference figures

Each bar is the value of a point minus the value of the point with the
variant `baseline` and the same load, in the same run. A bar to the left
of 0 means that the variant is faster. The three panels give the p50,
the p99 and the max. Each panel has its own x scale. The colour is the
load of the point. The menu above the figure selects the metric.

A grey band is the range that a check of this study expects, in the
same unit. If a band has one open end, it comes from an `order` check.
The manual checks are in the table of the checks only. The table below
each figure gives the same differences in cycles.

## IRQ entry (irq-kconfig-1m)

H1 a expects 0..10 cycles less for `speed-opt`. H2 a and b expect
0..10 cycles more for `assert` and `assert-spin`, and H2 c and d a
higher max. H3 a expects no change for `fpu-sharing` (+-2 cycles). All
checks of a range use the `irq` p50.

In [ ]:
VARIANTS = ['speed-opt', 'assert', 'assert-spin', 'fpu-sharing']
if nbfig.built(inst, "irq-kconfig-1m"):
    d = diffs("irq-kconfig-1m")
    sel = d[d.variant.isin(VARIANTS)]
    nbfig.diff_fig(sel, "irq-kconfig-1m: each variant minus the baseline of the same load",
                   limits=nbfig.check_limits(inst, "irq-kconfig-1m", d)).show()

In [ ]:
nbfig.diff_table(sel) if nbfig.built(inst, "irq-kconfig-1m") else None

## ISR to thread (wake-kconfig-1m)

H1 b expects a ratio of 0.75..0.90 to the baseline for `speed-opt`. H2 e
expects a ratio of 1.05..1.15 for `assert`. H2 f expects 100..150 cycles
more for `assert-spin` than for `assert`. H3 b expects 0..30 cycles
more for `fpu-sharing`. All checks use the `isr2thr` p50.

In [ ]:
VARIANTS = ['speed-opt', 'assert', 'assert-spin', 'fpu-sharing']
if nbfig.built(inst, "wake-kconfig-1m"):
    d = diffs("wake-kconfig-1m")
    sel = d[d.variant.isin(VARIANTS)]
    nbfig.diff_fig(sel, "wake-kconfig-1m: each variant minus the baseline of the same load",
                   limits=nbfig.check_limits(inst, "wake-kconfig-1m", d)).show()

In [ ]:
nbfig.diff_table(sel) if nbfig.built(inst, "wake-kconfig-1m") else None

## Thread to thread (thread-kconfig-1m)

H1 c, H2 g, H2 h and H3 c are the checks of H1 b, H2 e, H2 f and H3 b
on the `t2t` p50.

In [ ]:
VARIANTS = ['speed-opt', 'assert', 'assert-spin', 'fpu-sharing']
if nbfig.built(inst, "thread-kconfig-1m"):
    d = diffs("thread-kconfig-1m")
    sel = d[d.variant.isin(VARIANTS)]
    nbfig.diff_fig(sel, "thread-kconfig-1m: each variant minus the baseline of the same load",
                   limits=nbfig.check_limits(inst, "thread-kconfig-1m", d)).show()

In [ ]:
nbfig.diff_table(sel) if nbfig.built(inst, "thread-kconfig-1m") else None

## Ratio to the baseline

H1 and H2 give the change of the wake paths in percent. The table gives
the value of each variant divided by the value of the baseline of the
same load, in the same run.

In [ ]:
rows = []
for e in ("irq-kconfig-1m", "wake-kconfig-1m", "thread-kconfig-1m"):
    if nbfig.built(inst, e):
        d = diffs(e)
        d = d[d.variant.isin(VARIANTS)]
        rows.append(d.assign(experiment=e, **{s: d["v_" + s] / d["b_" + s] for s in nbfig.DIFF_STATS}))
ratio = pd.concat(rows)
ratio.pivot_table(index=["experiment", "metric"], columns="point", values="p50", sort=False).round(3)

In [ ]:
ratio.pivot_table(index=["experiment", "metric"], columns="point", values=["p99", "max"], sort=False).round(3)

## The distributions

The CCDF gives the share of the samples of x or more, for the baseline
and the four variants. H2 c and d expect a higher `irq` max for
`assert` and `assert-spin`.

In [ ]:
if nbfig.built(inst, "irq-kconfig-1m"):
    nbfig.ccdf(nbfig.hists_of(inst, "irq-kconfig-1m", points=["baseline"] + VARIANTS), "irq-kconfig-1m: CCDF").show()

In [ ]:
if nbfig.built(inst, "wake-kconfig-1m"):
    nbfig.ccdf(nbfig.hists_of(inst, "wake-kconfig-1m", points=["baseline"] + VARIANTS), "wake-kconfig-1m: CCDF").show()

In [ ]:
if nbfig.built(inst, "thread-kconfig-1m"):
    nbfig.ccdf(nbfig.hists_of(inst, "thread-kconfig-1m", points=["baseline"] + VARIANTS), "thread-kconfig-1m: CCDF").show()